# KazGPT V4 — Training on Google Colab A100

**Модель:** Qwen2.5-7B-Instruct + QLoRA  
**Датасет:** KazQAD (ISSAI NU) + kk-Wikipedia + multidomain (~25k примеров)  
**Время:** ~2–3 часа на A100 40GB  
**Результат:** GGUF адаптер → Ollama → KazGPT Spring Boot backend

### Перед запуском:
1. Runtime → Change runtime type → **A100 GPU**
2. Вставь свой HuggingFace токен в ячейку #3 (`HF_TOKEN`)
3. Запусти все ячейки по порядку (Runtime → Run all)

In [ ]:
# ─── Ячейка 1: Установка зависимостей + монтирование Drive ───────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/KazGPT_V4'
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/adapter',     exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/gguf',        exist_ok=True)
print(f'Drive path: {DRIVE_PATH}')

# unsloth даёт 2x скорость + меньше VRAM на A100
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece
print('Packages installed.')

In [ ]:
# ─── Ячейка 2: Проверка GPU ───────────────────────────────────────────────────
import torch

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()

print(f'GPU   : {gpu}')
print(f'VRAM  : {vram_gb:.1f} GB')
print(f'BF16  : {bf16_ok}')

if vram_gb < 20:
    print('\n⚠️  VRAM < 20GB — 7B модель может не влезть. Смени runtime на A100.')
else:
    print('\n✅  Готово к обучению 7B модели.')

In [ ]:
# ─── Ячейка 3: Конфигурация ────────────────────────────────────────────────────
# ⚠️  ЗАПОЛНИ HF_TOKEN: https://huggingface.co/settings/tokens (read access достаточно)
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX'

# Модель
MODEL_ID    = 'Qwen/Qwen2.5-7B-Instruct'   # 7B — влезает в A100 40GB с 4-bit QLoRA
MAX_SEQ     = 2048                           # контекст (V3 был 768, теперь полный)

# LoRA
LORA_RANK   = 64
LORA_ALPHA  = 128
LORA_DROP   = 0.05

# Обучение
BATCH_SIZE  = 2
GRAD_ACCUM  = 8    # effective_batch = 16
EPOCHS      = 3
LR          = 2e-4
WARMUP      = 0.05

# Датасет — сколько примеров взять из каждого источника
MAX_KAZQAD      = 12000   # KazQAD — весь (12k) — главный источник качества
MAX_WIKI        = 15000   # kk-Wikipedia — авто Q&A из первых предложений
MAX_MULTIDOMAIN = 5000    # multidomain фоллбек (если wiki недоступен)

# Пути
OUT_DIR         = f'{DRIVE_PATH}/checkpoints'
ADAPTER_DIR     = f'{DRIVE_PATH}/adapter'
GGUF_DIR        = f'{DRIVE_PATH}/gguf'

# System prompt — тот же что в application.yml
SYSTEM_PROMPT = (
    'Сен — KazGPT, Қазақстан үшін жасалған жергілікті жасанды интеллект. '
    'АБСОЛЮТТІ ЕРЕЖЕ: Барлық жауаптарыңды ТЕК ҚАЗАҚ ТІЛІНДЕ жаз. '
    'Сыпайы, кәсіби бол. Жауап қысқа: 1–4 сөйлем.'
)

print('Config OK.')
print(f'  Model  : {MODEL_ID}')
print(f'  Epochs : {EPOCHS}')
print(f'  LoRA r : {LORA_RANK}')
print(f'  MaxSeq : {MAX_SEQ}')
print(f'  EffBatch: {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
# ─── Ячейка 4: HuggingFace логин ─────────────────────────────────────────────
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK.')

In [ ]:
# ─── Ячейка 5: Загрузка и подготовка датасета ────────────────────────────────
import random, json
from datasets import load_dataset
random.seed(42)

def to_chatml(question: str, answer: str) -> dict:
    """ChatML формат с system prompt — совместим с Qwen2.5 instruct."""
    return {
        'text': (
            f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n{question.strip()}<|im_end|>\n'
            f'<|im_start|>assistant\n{answer.strip()}<|im_end|>'
        )
    }

def is_valid(q: str, a: str) -> bool:
    return bool(q and a and len(q.strip()) > 5 and len(a.strip()) > 10)

all_records = []

# ── Источник 1: KazQAD (ISSAI NU) — лучший, чистые QA на казахском ──────────
print('=> Загружаю issai/kazqad ...')
try:
    kqd = load_dataset('issai/kazqad', split='train')
    kqd_records = []
    for item in kqd:
        q = item.get('question') or item.get('query') or ''
        ans = item.get('answers') or item.get('answer') or ''
        if isinstance(ans, dict):
            texts = ans.get('text') or []
            a = texts[0] if texts else ''
        elif isinstance(ans, list):
            a = ans[0] if ans else ''
        else:
            a = str(ans)
        if is_valid(q, a):
            kqd_records.append(to_chatml(q, a))
        if len(kqd_records) >= MAX_KAZQAD:
            break
    all_records += kqd_records
    print(f'   [+] {len(kqd_records)} примеров из KazQAD')
except Exception as e:
    print(f'   [!] KazQAD: {e}')
    print('       Проверь HF_TOKEN и доступ к issai/kazqad')

# ── Источник 2: kk-Wikipedia — автогенерация Q&A из первых предложений ───────
print('=> Загружаю kk-Wikipedia ...')
try:
    wiki = load_dataset('wikimedia/wikipedia', '20231101.kk',
                        split='train', streaming=True, trust_remote_code=True)
    wiki_records = []
    templates = [
        '{title} деген не?',
        '{title} туралы қысқаша айтып бер.',
        '{title} жайында мәлімет бер.',
        '{title} дегеніміз не?',
    ]
    for item in wiki:
        title = (item.get('title') or '').strip()
        text  = (item.get('text')  or '').strip()
        if not title or not text:
            continue
        first = text.split('.')[0].strip()
        if len(first) < 30 or len(first) > 500:
            continue
        q = random.choice(templates).format(title=title)
        wiki_records.append(to_chatml(q, first + '.'))
        if len(wiki_records) >= MAX_WIKI:
            break
    all_records += wiki_records
    print(f'   [+] {len(wiki_records)} примеров из kk-Wikipedia')
except Exception as e:
    print(f'   [!] Wikipedia: {e} — пробую multidomain фоллбек...')
    try:
        md = load_dataset('kz-transformers/multidomain-kazakh-dataset',
                          split='train', streaming=True)
        md_records = []
        for item in md:
            text = (item.get('text') or item.get('content') or '').strip()
            if len(text) < 100:
                continue
            mid = len(text) // 2
            md_records.append(to_chatml(text[:mid], text[mid:]))
            if len(md_records) >= MAX_MULTIDOMAIN:
                break
        all_records += md_records
        print(f'   [+] {len(md_records)} примеров из multidomain')
    except Exception as e2:
        print(f'   [!] multidomain: {e2}')

# ── Финальное перемешивание и разбивка ───────────────────────────────────────
random.shuffle(all_records)
split = int(len(all_records) * 0.95)
train_data = all_records[:split]
valid_data = all_records[split:]

print(f'\nИтого:')
print(f'  Всего   : {len(all_records)}')
print(f'  Train   : {len(train_data)}')
print(f'  Valid   : {len(valid_data)}')

In [ ]:
# ─── Ячейка 6: Загрузка модели через unsloth ─────────────────────────────────
from unsloth import FastLanguageModel
import torch

print(f'Загружаю {MODEL_ID} с 4-bit QLoRA ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ,
    dtype           = None,          # auto: BF16 на A100
    load_in_4bit    = True,
    token           = HF_TOKEN,
)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'Модель загружена. VRAM занято: {vram_used:.1f} GB')

In [ ]:
# ─── Ячейка 7: LoRA адаптер ───────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROP,
    bias            = 'none',
    use_gradient_checkpointing = 'unsloth',  # экономит VRAM
    random_state    = 42,
    use_rslora      = False,
)
model.print_trainable_parameters()

In [ ]:
# ─── Ячейка 8: Обучение ───────────────────────────────────────────────────────
import time
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)

bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = valid_ds,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ,
    packing         = True,   # упаковывает короткие примеры → быстрее
    args = TrainingArguments(
        output_dir                  = OUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP,
        learning_rate               = LR,
        lr_scheduler_type           = 'cosine',
        fp16                        = not bf16,
        bf16                        = bf16,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        logging_steps               = 25,
        report_to                   = 'none',
        seed                        = 42,
        dataloader_num_workers      = 2,
    ),
)

print('═' * 50)
print('Обучение начинается...')
print(f'Train: {len(train_ds)} | Valid: {len(valid_ds)}')
print('═' * 50)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 3600

print(f'\n✅  Обучение завершено за {elapsed:.2f} ч')
print(f'   Best eval_loss: {trainer.state.best_metric:.4f}')

In [ ]:
# ─── Ячейка 9: Сохранение LoRA адаптера на Drive ─────────────────────────────
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Адаптер сохранён: {ADAPTER_DIR}')
!ls -lh {ADAPTER_DIR}

In [ ]:
# ─── Ячейка 10: Быстрая проверка качества ────────────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def ask(question: str) -> str:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = 256,
            temperature        = 0.27,
            top_p              = 0.85,
            repetition_penalty = 1.15,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

test_questions = [
    'Сәлем! Қалың қалай?',
    'Қазақстанның астанасы қандай?',
    'Нейрондық желі деген не?',
    'Привет, как дела?',
    'What is machine learning?',
]

print('═' * 50)
print('Проверка качества V4:')
print('═' * 50)
for q in test_questions:
    a = ask(q)
    print(f'Q: {q}')
    print(f'A: {a}')
    print('─' * 40)

In [ ]:
# ─── Ячейка 11: Экспорт в GGUF для Ollama ────────────────────────────────────
# Q4_K_M — оптимальный баланс качество/размер для Ollama
print('Экспортирую в GGUF Q4_K_M...')
print('(Это займёт 5–15 минут)')

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',
)

gguf_files = !ls -lh {GGUF_DIR}/*.gguf 2>/dev/null
print('GGUF файлы:')
for f in gguf_files:
    print(f'  {f}')
print(f'\n✅  Готово! Скачай файл из: {GGUF_DIR}')

## После обучения — подключение к KazGPT (локально)

### 1. Скачай GGUF из Google Drive
Папка: `KazGPT_V4/gguf/` → файл `*.gguf` (~4–5 GB)

### 2. Создай Modelfile для Ollama
```
FROM C:\путь\к\файлу\kazgpt-v4.gguf

PARAMETER temperature 0.27
PARAMETER top_p 0.85
PARAMETER repeat_penalty 1.15
PARAMETER num_predict 512
```

### 3. Зарегистрируй модель в Ollama
```bash
ollama create kazgpt-v4 -f Modelfile
ollama run kazgpt-v4 "Сәлем!"
```

### 4. Обнови application.yml
```yaml
kazgpt:
  models:
    base:
      url: http://localhost:11434
      name: kazgpt-v4        # ← сюда
      runtime: ollama
```

### 5. Перезапусти Spring Boot
```bash
cd backend && mvn spring-boot:run
```